# Titanic 
## Score: .78468

In [35]:
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd()
DATA = ROOT / "titanic"
RARE = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}


In [36]:
def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()

    ticket_vc = pd.concat([tr["Ticket"], te["Ticket"]], ignore_index=True).value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]

    for pcl in (1, 2, 3):
        m = tr.loc[tr["Pclass"] == pcl, "Fare"].median()
        if pd.notna(m):
            tr.loc[tr["Pclass"] == pcl, "Fare"] = tr.loc[tr["Pclass"] == pcl, "Fare"].fillna(m)
            te.loc[te["Pclass"] == pcl, "Fare"] = te.loc[te["Pclass"] == pcl, "Fare"].fillna(m)
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        pcl = pd.to_numeric(out["Pclass"], errors="coerce").fillna(3).astype(int)
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["SexFemale"] = (out["Sex"] == "female").astype(int)
        out["IsChild"] = (out["Age"] < 16).astype(int)
        out["FemaleOrChild"] = ((out["Sex"] == "female") | (out["Age"] < 16)).astype(int)
        out["Deck"] = out["Cabin"].apply(lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U")
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgePclass"] = out["Age"] * pcl.astype(float)
        out["AgeBin"] = pd.cut(out["Age"], bins=[0, 12, 18, 35, 60, 200], labels=["c", "t", "y", "a", "s"]).astype(str)
        out["Surname"] = out["Name"].str.split(",").str[0].str.strip()
        out["TicketKey"] = out["Ticket"].astype(str)
        out["FamilyKey"] = out["Surname"] + "_" + out["FamilySize"].astype(str)
        out["SexClassAge"] = out["Sex"].astype(str) + "_" + out["Pclass"].astype(str) + "_" + out["AgeBin"].astype(str)
        out["TicketPrefix"] = out["Ticket"].astype(str).str.replace(r"[0-9./ ]", "", regex=True)
        out["TicketPrefix"] = out["TicketPrefix"].replace("", "NONE")
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin", "Surname", "TicketKey", "FamilyKey", "SexClassAge", "TicketPrefix"]:
            out[c] = out[c].astype(str)
        return out

    X = feats(tr)
    X_test = feats(te)
    return X, y, X_test


def add_oof_target_encoding(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, cols, n_splits=5, alpha=15.0):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y_s = pd.Series(y)
    global_mean = y_s.mean()
    X_new = X.copy()
    X_test_new = X_test.copy()

    for col in cols:
        oof_col = np.zeros(len(X), dtype=float)
        for tr_i, va_i in cv.split(X, y):
            tr_c = X.iloc[tr_i][col]
            y_tr = y_s.iloc[tr_i]
            stat = y_tr.groupby(tr_c).agg(["mean", "count"])
            smooth = (stat["mean"] * stat["count"] + global_mean * alpha) / (stat["count"] + alpha)
            oof_col[va_i] = X.iloc[va_i][col].map(smooth).fillna(global_mean).to_numpy()
        full_stat = y_s.groupby(X[col]).agg(["mean", "count"])
        full_smooth = (full_stat["mean"] * full_stat["count"] + global_mean * alpha) / (full_stat["count"] + alpha)
        X_new[col + "_te"] = oof_col
        X_test_new[col + "_te"] = X_test[col].map(full_smooth).fillna(global_mean).to_numpy()
    return X_new, X_test_new


def add_oof_count_gated_prior(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, col: str, alpha: float, min_count: int):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    y_s = pd.Series(y)
    global_mean = float(y_s.mean())

    oof_col = np.zeros(len(X), dtype=float)
    for tr_i, va_i in cv.split(X, y):
        tr_c = X.iloc[tr_i][col]
        y_tr = y_s.iloc[tr_i]
        stat = y_tr.groupby(tr_c).agg(["mean", "count"])
        smooth = (stat["mean"] * stat["count"] + global_mean * alpha) / (stat["count"] + alpha)
        smooth = smooth.where(stat["count"] >= min_count, global_mean)
        oof_col[va_i] = X.iloc[va_i][col].map(smooth).fillna(global_mean).to_numpy()

    full_stat = y_s.groupby(X[col]).agg(["mean", "count"])
    full_smooth = (full_stat["mean"] * full_stat["count"] + global_mean * alpha) / (full_stat["count"] + alpha)
    full_smooth = full_smooth.where(full_stat["count"] >= min_count, global_mean)
    test_col = X_test[col].map(full_smooth).fillna(global_mean).to_numpy()
    return oof_col, test_col


train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
pid = test_raw["PassengerId"].values
X, y, X_test = build_xy(train_raw, test_raw)
cat_cols = ["Sex", "Embarked", "Title", "Deck", "Pclass", "AgeBin", "Surname", "TicketKey", "FamilyKey", "SexClassAge", "TicketPrefix"]
te_cols = ["Title", "Deck", "Embarked", "Pclass", "AgeBin", "Surname", "SexClassAge", "TicketPrefix"]
X_te, X_test_te = add_oof_target_encoding(X, y, X_test, te_cols, n_splits=5, alpha=15.0)

# EDA-backed soft priors from group keys with support gating.
X_te["TicketKey_prior"], X_test_te["TicketKey_prior"] = add_oof_count_gated_prior(
    X, y, X_test, col="TicketKey", alpha=20.0, min_count=2
)
X_te["FamilyKey_prior"], X_test_te["FamilyKey_prior"] = add_oof_count_gated_prior(
    X, y, X_test, col="FamilyKey", alpha=24.0, min_count=2
)

hgb_features = [
    "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "AgePclass",
    "TicketGroupSize", "FarePerPerson", "HasCabin",
    "SexFemale", "IsChild", "FemaleOrChild",
    "TicketKey_prior", "FamilyKey_prior",
] + [c + "_te" for c in te_cols]

imp = SimpleImputer(strategy="median")
X_hgb = imp.fit_transform(X_te[hgb_features])
X_test_hgb = imp.transform(X_test_te[hgb_features])


In [37]:
def fit_fold_models(Xc_tr, y_tr, Xh_tr, Xc_va, y_va, Xh_va, seed, sample_weight=None):
    """One CatBoost+HGB fit per CV fold (early stopping); same models for val + test."""
    cb1 = CatBoostClassifier(
        iterations=20000,
        learning_rate=0.016,
        depth=7,
        l2_leaf_reg=3.0,
        random_seed=seed,
        thread_count=-1,
        verbose=False,
    )
    cb1.fit(
        Xc_tr,
        y_tr,
        cat_features=cat_cols,
        sample_weight=sample_weight,
        eval_set=(Xc_va, y_va),
        early_stopping_rounds=320,
        verbose=False,
    )

    cb2 = CatBoostClassifier(
        iterations=26000,
        learning_rate=0.012,
        depth=9,
        l2_leaf_reg=4.5,
        random_seed=seed + 97,
        thread_count=-1,
        verbose=False,
    )
    cb2.fit(
        Xc_tr,
        y_tr,
        cat_features=cat_cols,
        sample_weight=sample_weight,
        eval_set=(Xc_va, y_va),
        early_stopping_rounds=380,
        verbose=False,
    )

    hgb = HistGradientBoostingClassifier(
        learning_rate=0.02,
        max_iter=1400,
        max_depth=8,
        min_samples_leaf=6,
        l2_regularization=0.35,
        random_state=seed + 13,
    )
    hgb.fit(Xh_tr, y_tr, sample_weight=sample_weight)

    p1_va = cb1.predict_proba(Xc_va)[:, 1]
    p2_va = cb2.predict_proba(Xc_va)[:, 1]
    p3_va = hgb.predict_proba(Xh_va)[:, 1]
    p1_te = cb1.predict_proba(X_test)[:, 1]
    p2_te = cb2.predict_proba(X_test)[:, 1]
    p3_te = hgb.predict_proba(X_test_hgb)[:, 1]
    return p1_va, p2_va, p3_va, p1_te, p2_te, p3_te


def optimize_blend(oof_cb1, oof_cb2, oof_hgb, y_true):
    best_raw_acc, best_w1, best_w2, best_t_raw = -1.0, 0.33, 0.33, 0.5
    for w1 in np.linspace(0.22, 0.88, 67):
        for w2 in np.linspace(0.05, 0.58, 54):
            if w1 + w2 >= 0.98:
                continue
            w3 = 1.0 - w1 - w2
            p = w1 * oof_cb1 + w2 * oof_cb2 + w3 * oof_hgb
            for t in np.linspace(0.40, 0.62, 111):
                a = accuracy_score(y_true, (p >= t).astype(int))
                if a > best_raw_acc:
                    best_raw_acc, best_w1, best_w2, best_t_raw = a, float(w1), float(w2), float(t)
    best_w3 = 1.0 - best_w1 - best_w2
    return best_raw_acc, (best_w1, best_w2, best_w3), best_t_raw


def fit_pseudo_models(Xc_aug, y_aug, Xh_aug, w_aug, seed):
    """Train once on augmented data; return probs for train X and test X."""
    idx = np.arange(len(y_aug))
    tr_i, va_i = train_test_split(idx, test_size=0.12, stratify=y_aug, random_state=SEED)
    Xc_tr, Xc_va = Xc_aug.iloc[tr_i], Xc_aug.iloc[va_i]
    Xh_tr, Xh_va = Xh_aug[tr_i], Xh_aug[va_i]
    y_tr, y_va = y_aug[tr_i], y_aug[va_i]
    sw_tr = w_aug[tr_i]

    cb1 = CatBoostClassifier(
        iterations=12000,
        learning_rate=0.018,
        depth=7,
        l2_leaf_reg=3.0,
        random_seed=seed,
        thread_count=-1,
        verbose=False,
    )
    cb1.fit(
        Xc_tr,
        y_tr,
        cat_features=cat_cols,
        sample_weight=sw_tr,
        eval_set=(Xc_va, y_va),
        early_stopping_rounds=280,
        verbose=False,
    )

    cb2 = CatBoostClassifier(
        iterations=16000,
        learning_rate=0.013,
        depth=9,
        l2_leaf_reg=4.5,
        random_seed=seed + 97,
        thread_count=-1,
        verbose=False,
    )
    cb2.fit(
        Xc_tr,
        y_tr,
        cat_features=cat_cols,
        sample_weight=sw_tr,
        eval_set=(Xc_va, y_va),
        early_stopping_rounds=320,
        verbose=False,
    )

    hgb = HistGradientBoostingClassifier(
        learning_rate=0.02,
        max_iter=1200,
        max_depth=8,
        min_samples_leaf=6,
        l2_regularization=0.35,
        random_state=seed + 13,
    )
    hgb.fit(Xh_tr, y_tr, sample_weight=sw_tr)

    p1_tr = cb1.predict_proba(X)[:, 1]
    p2_tr = cb2.predict_proba(X)[:, 1]
    p3_tr = hgb.predict_proba(X_hgb)[:, 1]
    p1_te = cb1.predict_proba(X_test)[:, 1]
    p2_te = cb2.predict_proba(X_test)[:, 1]
    p3_te = hgb.predict_proba(X_test_hgb)[:, 1]
    return p1_tr, p2_tr, p3_tr, p1_te, p2_te, p3_te


def run_cv_with_pseudo():
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_cb1 = np.zeros(len(y), dtype=float)
    oof_cb2 = np.zeros(len(y), dtype=float)
    oof_hgb = np.zeros(len(y), dtype=float)
    p_cb1_t = np.zeros(len(X_test), dtype=float)
    p_cb2_t = np.zeros(len(X_test), dtype=float)
    p_hgb_t = np.zeros(len(X_test), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        print("fold", fold, "/5 training...", flush=True)
        Xc_tr, Xc_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        Xh_tr, Xh_va = X_hgb[tr_idx], X_hgb[va_idx]

        p1_va, p2_va, p3_va, p1_te, p2_te, p3_te = fit_fold_models(
            Xc_tr, y_tr, Xh_tr, Xc_va, y_va, Xh_va, seed=SEED + fold, sample_weight=None
        )
        oof_cb1[va_idx], oof_cb2[va_idx], oof_hgb[va_idx] = p1_va, p2_va, p3_va
        p_cb1_t += p1_te / cv.n_splits
        p_cb2_t += p2_te / cv.n_splits
        p_hgb_t += p3_te / cv.n_splits

    print("stage1 CV done, blending...", flush=True)
    raw_acc, w, t_raw = optimize_blend(oof_cb1, oof_cb2, oof_hgb, y)
    oof_lin = w[0] * oof_cb1 + w[1] * oof_cb2 + w[2] * oof_hgb
    test_lin = w[0] * p_cb1_t + w[1] * p_cb2_t + w[2] * p_hgb_t

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof_lin, y)
    oof_stage1 = iso.predict(oof_lin)
    test_stage1 = iso.predict(test_lin)

    lo, hi = 0.04, 0.96
    pseudo_mask = (test_stage1 <= lo) | (test_stage1 >= hi)
    pseudo_y = (test_stage1[pseudo_mask] >= 0.5).astype(int)
    pseudo_w = np.full(int(pseudo_mask.sum()), 0.45, dtype=float)

    X_cat_aug = pd.concat([X, X_test.loc[pseudo_mask]], axis=0, ignore_index=True)
    X_hgb_aug = np.vstack([X_hgb, X_test_hgb[pseudo_mask]])
    y_aug = np.concatenate([y, pseudo_y])
    w_aug = np.concatenate([np.ones(len(y), dtype=float), pseudo_w])

    print("pseudo stage2 (n=%d)..." % int(pseudo_mask.sum()), flush=True)
    p1b, p2b, p3b, t1b, t2b, t3b = fit_pseudo_models(
        X_cat_aug, y_aug, X_hgb_aug, w_aug, seed=SEED + 200
    )

    oof_pseudo = w[0] * p1b + w[1] * p2b + w[2] * p3b
    test_pseudo = w[0] * t1b + w[1] * t2b + w[2] * t3b

    iso2 = IsotonicRegression(out_of_bounds="clip")
    iso2.fit(oof_pseudo, y)
    oof_final = iso2.predict(oof_pseudo)
    test_final = iso2.predict(test_pseudo)

    best_acc, best_t = -1.0, 0.5
    for t in np.linspace(0.05, 0.95, 181):
        a = accuracy_score(y, (oof_final >= t).astype(int))
        if a > best_acc:
            best_acc, best_t = a, float(t)

    print(
        "cv v9 pseudo",
        "acc",
        round(best_acc, 5),
        "t",
        round(best_t, 3),
        "w",
        [round(x, 3) for x in w],
        "pseudo_n",
        int(pseudo_mask.sum()),
        "raw_stage1_acc",
        round(raw_acc, 5),
    )
    print(confusion_matrix(y, (oof_final >= best_t).astype(int)))

    return {
        "mode": "v9_pseudo_confident",
        "acc": best_acc,
        "t": best_t,
        "w": w,
        "pseudo_n": int(pseudo_mask.sum()),
        "oof_prob": oof_final,
        "test_prob": test_final,
    }


safe = run_cv_with_pseudo()

for cutoff in (12, 14, 16):
    m = train_raw["Age"].fillna(999) < cutoff
    rate = float(train_raw.loc[m, "Survived"].mean()) if m.any() else float("nan")
    print("train survival age<%d: n=%d, rate=%.3f" % (cutoff, int(m.sum()), rate))


fold 1 /5 training...
fold 2 /5 training...
fold 3 /5 training...
fold 4 /5 training...
fold 5 /5 training...
stage1 CV done, blending...
pseudo stage2 (n=46)...
cv v9 pseudo acc 0.96857 t 0.14 w [0.27, 0.43, 0.3] pseudo_n 46 raw_stage1_acc 0.85297
[[532  17]
 [ 11 331]]
train survival age<12: n=68, rate=0.574
train survival age<14: n=71, rate=0.592
train survival age<16: n=83, rate=0.590


In [38]:
prob = safe["test_prob"].copy()
pred = (prob >= safe["t"]).astype(int)

submission = pd.DataFrame({"PassengerId": pid, "Survived": pred})
submission.to_csv(ROOT / "submission.csv", index=False)
print("wrote", ROOT / "submission.csv", safe.get("mode", ""))
submission.head(10)


wrote c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv v9_pseudo_confident


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
